# MAPPO Autocurriculum

Train one JAX MAPPO policy on an autocurriculum env that grows from `4x4` through `50x50` inside each episode after six deliveries per active stage.


In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
{"project_root": PROJECT_ROOT, **runtime_status}


In [ ]:
import importlib

import jax

from ant_byte_env import notebook_workflows as workflows
from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


## Quick Smoke Run

Run one tiny training job to confirm imports, kernel state, and JAX execution before starting the longer autocurriculum run.


In [ ]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics


## Autocurriculum Settings

The experiment config owns the env shape, food-source contract, and PPO defaults; this notebook only chooses run paths and logging switches.


In [ ]:
AUTOCURRICULUM_CONFIG = PROJECT_ROOT / "experiments" / "autocurriculum.json"
experiment = workflows.load_jax_experiment(AUTOCURRICULUM_CONFIG)

RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "autocurriculum"
MEDIA_DIR = RUN_DIR / "media"
GLOBAL_UPDATE_CAP = int(experiment.metadata.get("global_update_cap", 10000))
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
WANDB_PROJECT = "cool-antz"
WANDB_ENTITY = None
WANDB_GROUP = "autocurriculum_50x50"
WANDB_MODE = "online"

COMMON_ARGS = workflows.config_common_args(
    experiment.args,
    exclude=workflows.AUTOCURRICULUM_ARG_EXCLUDES,
)
COMMON_ARGS += ["--wandb-mode", WANDB_MODE, "--wandb-group", WANDB_GROUP]
if WANDB_PROJECT is not None:
    COMMON_ARGS += ["--wandb-project", WANDB_PROJECT]
if WANDB_ENTITY is not None:
    COMMON_ARGS += ["--wandb-entity", WANDB_ENTITY]
UPDATE_TIMESTEPS = workflows.update_timesteps(
    num_envs=int(experiment.args["num_envs"]),
    num_steps=int(experiment.args["num_steps"]),
)

{
    "config": AUTOCURRICULUM_CONFIG,
    "updates": GLOBAL_UPDATE_CAP,
    "update_timesteps": UPDATE_TIMESTEPS,
    "wandb_group": WANDB_GROUP,
}


## Train Autocurriculum Policy


In [ ]:
autocurriculum_result = workflows.run_autocurriculum_training(
    run_dir=RUN_DIR,
    common_args=COMMON_ARGS,
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
)
FINAL_CHECKPOINT_PATH = autocurriculum_result["checkpoint_path"]
autocurriculum_result


## Optional Render and Vault


In [ ]:
rollout_result = workflows.render_autocurriculum_rollout(
    run_dir=RUN_DIR,
    checkpoint_path=FINAL_CHECKPOINT_PATH,
    media_dir=MEDIA_DIR,
    global_update_cap=GLOBAL_UPDATE_CAP,
    tile_size=ROLLOUT_TILE_SIZE,
)
rollout_result
